In [17]:
from pytorch_forecasting import TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.data.encoders import TorchNormalizer
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

In [19]:
df = pd.read_csv('../data/preprocessed/crypto_data_proc.csv')
print(f"Total rows: {len(df):,}")
print(f"Tokens: {df['token'].nunique()}")
print(f"Time range: {df['time_idx'].min()} to {df['time_idx'].max()}")
df.head()

Total rows: 1,011,769
Tokens: 30
Time range: 0 to 52029


,time_idx,token,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
0,0,1INCH,2.558157,-0.138144,2.409150,-0.030149,16.926395,0.001111,0.007636,0.025986,1.000000,6.123234e-17,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.045938
1,1,1INCH,-0.026367,0.070677,0.059331,0.045938,16.874238,0.001031,-0.001646,0.025916,0.965926,-2.588190e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.003933
2,2,1INCH,0.045316,-0.021816,0.043149,-0.003933,16.677244,0.001167,0.006651,0.026059,0.866025,-5.000000e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.008800
3,3,1INCH,-0.008165,-0.026874,0.038948,0.008800,16.110003,0.010799,0.010399,0.026049,0.707107,-7.071068e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.048203
4,4,1INCH,0.009949,0.079119,0.027554,0.048203,16.923341,0.001701,0.003810,0.026104,0.500000,-8.660254e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.044002


# Train/Val/Test Split (Per Token)
Since each token has its own time series with different lengths, we split each token's data individually using the same percentage thresholds.

In [20]:
# Split each token's time series: 70% train, 15% val, 15% test
def split_by_token(group, train_pct=0.70, val_pct=0.15):
  """Split a single token's time series into train/val/test"""
  max_idx = group['time_idx'].max()
  train_cutoff = int(train_pct * max_idx)
  val_cutoff = int((train_pct + val_pct) * max_idx)
  
  group['split'] = 'test'  # default
  group.loc[group['time_idx'] <= train_cutoff, 'split'] = 'train'
  group.loc[(group['time_idx'] > train_cutoff) & (group['time_idx'] <= val_cutoff), 'split'] = 'val'
  
  return group

# Apply split to each token
df = df.groupby('token', group_keys=False).apply(split_by_token)

# Verify splits
print("Split percentages:")
print(df['split'].value_counts(normalize=True) * 100)

Split percentages:
split
train    69.999773
test     15.000756
val      14.999471
Name: proportion, dtype: float64


/var/folders/zf/pd74vy996kg6l6dpbrpqm1jr0000gn/T/ipykernel_93390/2907654520.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('token', group_keys=False).apply(split_by_token)


In [21]:
# Create separate dataframes for each split
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

print(f"Train: {len(train_df):,} rows")
print(f"Val: {len(val_df):,} rows")
print(f"Test: {len(test_df):,} rows")

Train: 708,236 rows
Val: 151,760 rows
Test: 151,773 rows


# Create TimeSeriesDataSet with RobustGroupNormalizer

In [22]:
# Define parameters
max_encoder_length = 168  # 1 week of hourly data
max_prediction_length = 1  # Predict 1 step ahead (next_close_log)
min_encoder_length = 24   # Minimum 1 day of history

# Financial features that need RobustGroupNormalizer (log returns)
financial_features = [
  "open_log", "high_log", "low_log", "close_log", "volume_log",
  "btc_close_log", "eth_close_log", "eth_btc_ratio"
]

# Cyclical temporal features (already bounded [-1, 1], no normalization needed)
temporal_features = [
  "time_idx",
  "hour_sin", "hour_cos",
  "norm_day_sin", "norm_day_cos", 
  "weekday_sin", "weekday_cos",
  "month_sin", "month_cos",
]

# Create scalers dictionary for financial features
scalers = {
  "open_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "high_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "low_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "close_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "volume_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="standard", center=True)),
  "btc_close_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "eth_close_log": GroupNormalizer(groups=["token"], transformation=TorchNormalizer(method="robust", center=True)),
  "eth_btc_ratio": GroupNormalizer(groups=[], transformation=MinMaxScaler())  # Global
}

print(f"Encoder length: {max_encoder_length} timesteps")
print(f"Prediction length: {max_prediction_length} timesteps")
print(f"Financial features with RobustGroupNormalizer: {len(financial_features)}")

Encoder length: 168 timesteps
Prediction length: 1 timesteps
Financial features with RobustGroupNormalizer: 8


In [24]:
# Create training TimeSeriesDataSet
training = TimeSeriesDataSet(
  train_df,
  time_idx="time_idx",
  target="next_close_log",
  group_ids=["token"],
  
  # Sequence lengths
  max_encoder_length=max_encoder_length,
  max_prediction_length=max_prediction_length,
  min_encoder_length=min_encoder_length,
  
  # Known features (available at prediction time)
  time_varying_known_reals=temporal_features,
  
  # Unknown features (only known historically, including all financial features)
  time_varying_unknown_reals=financial_features,
  
  # Categorical features
  static_categoricals=["token"],
  
  # Target normalization
  target_normalizer=GroupNormalizer(
      groups=["token"],
      transformation=TorchNormalizer(method="robust", center=True)
  ),
  
  # Apply RobustGroupNormalizer to all financial features
  scalers=scalers,
  
  # Additional parameters
  add_relative_time_idx=True,
  add_target_scales=True,
  add_encoder_length=True,
  allow_missing_timesteps=False,
)

print("Training dataset created successfully!")
print(f"Number of samples: {len(training)}")

AttributeError: 'TorchNormalizer' object has no attribute 'setdefault'

In [ ]:
# Create validation dataset (inherits parameters from training)
validation = TimeSeriesDataSet.from_dataset(
  training,
  val_df,
  predict=True,
  stop_randomization=True
)

print("Validation dataset created successfully!")
print(f"Number of samples: {len(validation)}")